# Notebook 14 deterministic environment recovery

Purpose: restore the already registered deterministic FP32 backend before frozen worker preflight. This fixes the launch path environment, preserving scientific models, data, seeds, update budget, original clock and every exact-resume assertion. Inputs: frozen Notebook14 study binding, worker plans and initialization states. Outputs: hashed quarantine of failed software-only proofs, corrected full-state and probe certificates, backend records and detached recovered worker notebooks. Definition load takes seconds; two-worker preflight takes roughly two minutes; scientific training uses remaining original two-hour budget. Checkpoints remain `checkpoints/<run_id>/stages/worker*/`. Never reset the budget clock.

Exact recovery sequence: call `prepare_environment_recovery(ROOT, RUN_ID)` once; run `verify_recovered_preflight(ROOT,RUN_ID,worker)` for each assigned GPU; require both PASS; root restarts the original-deadline guard; call `launch_recovered_workers(ROOT,RUN_ID)`; resume the frozen coordinator/main notebook. No long training may start before both strict proofs pass. The tagged definition cell never launches by import.

The worker also selects its assigned current CUDA device before creating timing events. Tensor placement and event recording/synchronization must refer to the same GPU. Earlier software-only failed timing state is retained; scientific schedule and optimizer/model mathematics remain unchanged.

Software probe validation checkpoints are isolated under the assigned worker stage. The inherited single-worker validation function is copied at runtime with exactly two checkpoint-directory expressions replaced; all mathematical operations and assertions remain text-identical. Scientific probe extraction/fitting continues through unchanged registered exports and already isolates workers.

In [ ]:
"""Operational recovery: configure the already registered deterministic backend before preflight."""
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from pathlib import Path
import hashlib,json,time,types,sys
import nbformat,torch
ADAPTER_RELATIVE='experiments/14_routing_capacity/recoveries/deterministic_environment.ipynb'

def _workflow(root):
    root=Path(root).resolve()
    boot={'__file__':str(root/'fmm_jepa/bootstrap.ipynb')}
    for cell in nbformat.read(root/'fmm_jepa/bootstrap.ipynb',4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'bootstrap#ahr-export','exec'),boot)
    boot['install_notebook_package'](root)
    from fmm_jepa.bootstrap import TaggedNotebookLoader
    key='fmm14_recovery_original_workflow'
    if key not in sys.modules:
        path=root/'experiments/14_routing_capacity/workflow.ipynb';module=types.ModuleType(key);module.__file__=str(path);sys.modules[key]=module
        TaggedNotebookLoader(path).exec_module(module)
    return sys.modules[key]

def _checked(root,run_id):
    root=Path(root).resolve();workflow=_workflow(root);artifact=root/'artifacts/notebook_14'/run_id
    binding=workflow._read(artifact/'study_binding.json');workflow._verify(root,artifact,binding)
    if time.time()>=binding['budget_clock']['deadline_unix']:raise TimeoutError('Original two-hour deadline expired; never reset')
    return root,artifact,workflow,binding

def _adapter_record(root):
    path=Path(root)/ADAPTER_RELATIVE
    return {'path':str(path),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}

def _backend():
    return {'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
      'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,'tf32_cudnn':torch.backends.cudnn.allow_tf32,
      'cpu_threads':torch.get_num_threads(),'cublas_workspace':os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
      'cuda_current_device':torch.cuda.current_device()}

def configure_worker_environment(root,run_id,worker):
    root,artifact,workflow,binding=_checked(root,run_id)
    folder=artifact/'workers'/('worker'+str(worker));plan=workflow._read(folder/'plan.json')
    before=_backend();evaluation=workflow._load(root,'evaluation');evaluation.configure_environment(plan['arms'][0]['config']);torch.cuda.set_device(torch.device(plan['arms'][0]['config']['device']));after=_backend()
    expected={'deterministic_algorithms':True,'tf32_matmul':False,'tf32_cudnn':False,
      'cpu_threads':int(plan['arms'][0]['config'].get('cpu_threads',2)),'cublas_workspace':':4096:8','cuda_current_device':int(plan['arms'][0]['config']['device'].split(':')[1])}
    if after!=expected:raise RuntimeError('Registered deterministic backend not established')
    record={'status':'CONFIGURED','worker':worker,'backend_before':before,'backend_after':after,
      'adapter':_adapter_record(root),'plan_hash':workflow._hash(plan),'source_hashes':binding['source_hashes'],
      'original_clock':binding['budget_clock'],'updated_unix':time.time(),
      'scope':'Operational environment only. Frozen scientific source, model/data/masks/config/schedule/assertions unchanged.'}
    from fmm_jepa.training import atomic_json
    atomic_json(artifact/'recoveries/deterministic_environment'/('worker'+str(worker)+'_backend.json'),record)
    return record

def prepare_environment_recovery(root,run_id):
    """Quarantine only failed software proof state, retaining all evidence and immutable registrations."""
    root,artifact,workflow,binding=_checked(root,run_id)
    from fmm_jepa.training import atomic_json,digest_file
    from fmm_jepa.food101_training import run_writer
    folder=artifact/'recoveries/deterministic_environment';folder.mkdir(parents=True,exist_ok=True);receipt=folder/'quarantine.json'
    with run_writer(folder/'quarantine_writer'):
        if receipt.exists():
            result=workflow._read(receipt)
            if result['source_hashes']!=binding['source_hashes'] or result['original_clock']!=binding['budget_clock']:raise ValueError('Recovery registration changed')
            if result['adapter']!=_adapter_record(root):
                evolution=workflow._read(folder/'adapter_update.json')
                if evolution['initial_sha256']!=result['adapter']['sha256'] or evolution['current_sha256']!=_adapter_record(root)['sha256']:raise ValueError('Recovery adapter evolution differs')
            return result
        forbidden=[]
        for worker in range(2):
            stage='worker'+str(worker);part=artifact/'workers'/stage;plan=workflow._read(part/'plan.json')
            for arm in plan['arms']:
                scientific=root/'checkpoints'/run_id/'stages'/stage/'arms'/arm['id']
                if scientific.exists() and any(scientific.glob('*.pt')):forbidden.append(str(scientific))
                if (scientific/'manifest.json').exists():forbidden.append(str(scientific/'manifest.json'))
            for name in ('training_result.json','stage_training_summary.json','stage_evaluation.json','stage_evaluation_manifest.json','worker_result.json'):
                if (part/name).exists():forbidden.append(str(part/name))
        if forbidden:raise RuntimeError('Scientific state exists; this pretraining-only quarantine is forbidden: '+repr(forbidden))
        jobs=[]
        for worker in range(2):
            stage='worker'+str(worker);part=artifact/'workers'/stage
            for source,dest in [
                (part/'preflight',folder/'quarantined_artifacts'/stage/'preflight'),
                (root/'checkpoints'/run_id/'stages'/stage/'preflight',root/'checkpoints'/run_id/'recoveries/deterministic_environment'/'quarantined'/stage/'preflight'),
                (part/'stage_preflight.json',folder/'quarantined_artifacts'/stage/'stage_preflight.json'),
                (part/'preflight_result.json',folder/'quarantined_artifacts'/stage/'preflight_result.json')]:
                if source.exists():
                    if dest.exists():raise ValueError('Quarantine destination already exists without completed registration')
                    files=[source] if source.is_file() else sorted(p for p in source.rglob('*')if p.is_file())
                    records=[{'original':str(p),'relative':str(p.relative_to(source))if source.is_dir()else p.name,'sha256':digest_file(p),'bytes':p.stat().st_size}for p in files]
                    jobs.append({'source':str(source),'destination':str(dest),'files':records})
        for job in jobs:
            source,dest=Path(job['source']),Path(job['destination']);dest.parent.mkdir(parents=True,exist_ok=True);os.replace(source,dest)
            for record in job['files']:
                target=dest/record['relative'] if dest.is_dir() else dest
                if digest_file(target)!=record['sha256']:raise RuntimeError('Quarantined evidence checksum mismatch')
        result={'status':'PREPARED','adapter':_adapter_record(root),'source_hashes':binding['source_hashes'],'original_clock':binding['budget_clock'],
          'scientific_training_checkpoints':0,'scientific_quality_endpoints':0,'quarantined':jobs,'prepared_unix':time.time(),
          'diagnosis':'Failed preflight environment recorded deterministic_algorithms=False, cpu_threads12, TF32cuDNN=True. Prior passing actual software smoke recorded True,2,False. Equivalent data traces/config/losses; tiny floating-point tensor differences. Restore intended already exported deterministic environment before running frozen exact validator.',
          'policy':'No checkpoint assertion weakened; no frozen source/config/init/protocol/data/seed/update/deadline changed. Only failed software-validation state moved with verified hashes; all scientific arms remain pristine.'}
        atomic_json(receipt,result);return result

def _isolated_software_probe_validation(evaluation,root,artifact,plan,bundle,deadline):
    """Copy frozen validation with only worker-specific checkpoint directory names."""
    import ast
    source_path=Path(root)/'experiments/14_routing_capacity/evaluation.ipynb'
    sources='\n'.join(cell.source for cell in nbformat.read(source_path,4).cells if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]))
    tree=ast.parse(sources);node=next(node for node in tree.body if isinstance(node,ast.FunctionDef) and node.name=='verify_evaluation_smoke')
    original=ast.get_source_segment(sources,node)
    old="root/'checkpoints'/plan['run_id']/'validation'/"
    new="root/'checkpoints'/plan['run_id']/'stages'/plan['stage_id']/'validation'/"
    if original.count(old)!=2:raise ValueError('Expected exactly two unchanged software probe checkpoint expressions')
    isolated=original.replace(old,new)
    original_tree=ast.parse(original);isolated_tree=ast.parse(isolated)
    # Apart from the two literal checkpoint path expressions, every statement
    # and mathematical/assertion branch is required to remain text-identical.
    if isolated.replace(new,old)!=original:raise AssertionError('Validation change beyond checkpoint namespacing')
    namespace=dict(evaluation.__dict__);exec(compile(isolated,str(source_path)+'#operational-worker-checkpoint-namespace','exec'),namespace)
    from fmm_jepa.training import atomic_json
    record={'status':'INSTALLED_FOR_SOFTWARE_VALIDATION_ONLY','worker_stage':plan['stage_id'],
      'original_function_sha256':hashlib.sha256(original.encode()).hexdigest(),'operational_function_sha256':hashlib.sha256(isolated.encode()).hexdigest(),
      'replacement_count':2,'original_expression':old,'operational_expression':new,'all_math_and_assertions_text_identical':True,
      'scientific_function_export_file_changed':False,'plan_run_id_changed':False,'original_deadline':deadline}
    atomic_json(Path(root)/'artifacts/notebook_14'/plan['run_id']/'recoveries/deterministic_environment'/(plan['stage_id']+'_probe_namespace.json'),record)
    return namespace['verify_evaluation_smoke'](root,artifact,plan,bundle,deadline)

def verify_recovered_preflight(root,run_id,worker):
    """Actual frozen worker proof, no scientific training invocation."""
    root,artifact,workflow,binding=_checked(root,run_id);stage='worker'+str(worker);folder=artifact/'workers'/stage
    configure_worker_environment(root,run_id,worker)
    from fmm_jepa.training import atomic_json
    from fmm_jepa.food101_training import run_writer
    plan=workflow._read(folder/'plan.json');prep,runner,evaluation=(workflow._load(root,n)for n in('preparation','runner','evaluation'))
    bundle=prep.prepare_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=True)
    with run_writer(folder/'worker_lock'):
        runner.prepare_stage(root,folder,plan,bundle)
        full=runner.verify_stage_resume(root,folder,plan,bundle,binding['budget_clock']['deadline_unix'])
        if full['status']!='PASS':raise RuntimeError('Frozen actual-config exact resume failed')
        atomic_json(folder/'preflight_result.json',full)
        probes=_isolated_software_probe_validation(evaluation,root,folder,plan,bundle,binding['budget_clock']['deadline_unix'])
        if probes['status']!='PASS':raise RuntimeError('Frozen fixed-epoch probe integration/resume failed')
        atomic_json(folder/'evaluation_preflight_result.json',probes)
        workflow._verify(root,artifact,binding)
        receipt={'status':'PASS','worker':worker,'variants':plan['variants'],'backend':_backend(),'plan_hash':workflow._hash(plan),
          'source_hashes':binding['source_hashes'],'adapter':_adapter_record(root),'original_clock':binding['budget_clock'],
          'exact_resume_proof':str(folder/'preflight_result.json'),'probe_proof':str(folder/'evaluation_preflight_result.json'),
          'scope':'Frozen full-state bitwise four-versus-two-plus-two proof for all four actual architectures and actual assigned seed/data/schedule. Fixed20epoch real-feature/probe resume. No scientific training started.','completed_unix':time.time()}
        atomic_json(artifact/'recoveries/deterministic_environment'/(stage+'_validation.json'),receipt);return receipt

def _install_software_probe_validation(evaluation):
    if not getattr(evaluation,'_fmm14_operational_probe_namespace',False):
        def isolated(root,artifact,plan,bundle,deadline):
            return _isolated_software_probe_validation(evaluation,root,artifact,plan,bundle,deadline)
        evaluation.verify_evaluation_smoke=isolated
        evaluation._fmm14_operational_probe_namespace=True
    return evaluation

def run_recovered_worker(root,run_id,worker):
    configure_worker_environment(root,run_id,worker)
    workflow=_workflow(root);_install_software_probe_validation(workflow._load(root,'evaluation'))
    return workflow.run_worker(Path(root).resolve(),run_id,int(worker))

def launch_recovered_workers(root,run_id,WORKFLOW=None,adapterpath=None):
    """Root launches guard first and coordinator afterward; session names match frozen coordinator."""
    root,artifact,workflow,binding=_checked(root,run_id);workflow=workflow if WORKFLOW is None else WORKFLOW
    if adapterpath is not None and Path(adapterpath).resolve()!=(root/ADAPTER_RELATIVE).resolve():raise ValueError('Adapter source differs')
    for worker in range(2):
        proof=workflow._read(artifact/'recoveries/deterministic_environment'/('worker'+str(worker)+'_validation.json'))
        if proof.get('status')!='PASS' or proof['adapter']!=_adapter_record(root) or proof['original_clock']!=binding['budget_clock']:raise ValueError('Corrected frozen exact resume proof required before long training')
        modulecode="import types\nRECOVERY=types.ModuleType('fmm14_environment_recovery');RECOVERY.__file__=str(ROOT/"+repr(ADAPTER_RELATIVE)+")\nTaggedNotebookLoader(RECOVERY.__file__).exec_module(RECOVERY)\nRECOVERY.run_recovered_worker(ROOT,"+repr(run_id)+","+str(worker)+")"
        workflow._spawn(root,artifact,'fmm-routing-'+run_id+'-worker'+str(worker),modulecode,artifact/'recoveries/deterministic_environment'/('worker'+str(worker)+'_recovered.ipynb'))
    return {'status':'LAUNCHED_OR_ATTACHED','sessions':['fmm-routing-'+run_id+'-worker'+str(w)for w in range(2)],'original_clock':binding['budget_clock'],'adapter':_adapter_record(root)}
